# Оценка результатов A/B-теста интернет-магазина BitMotion Kit

**Автор:** Екатерина Штрейс  
**Дата:** 18 мая 2026 года

## Бизнес-контекст

Интернет-магазин BitMotion Kit протестировал упрощённую версию интерфейса, рассчитывая увеличить долю зарегистрированных пользователей, совершающих покупку. Анализ включает проверку качества эксперимента, очистку пересечений между тестами, формирование единого семидневного горизонта и статистическую оценку прироста конверсии.

Целевой эффект задан как увеличение конверсии минимум на 3 процентных пункта относительно базового уровня 30%.

## 1. Цель и дизайн исследования

Цель исследования — определить, корректно ли проведён A/B-тест и обеспечил ли новый интерфейс статистически подтверждённый рост конверсии в покупку не менее чем на 3 процентных пункта.

## 2. Загрузка и контроль качества данных

In [7]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')

events = pd.read_csv(
    'https://code.s3.yandex.net/datasets/ab_test_events.zip',
    parse_dates=['event_dt'],
    low_memory=False
)

In [8]:
display(participants.head())
display(events.head())

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [9]:
participants.info()
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [10]:
display(participants.isna().sum())
display(events.isna().sum())

user_id    0
group      0
ab_test    0
device     0
dtype: int64

user_id            0
event_dt           0
event_name         0
details       538264
dtype: int64

In [11]:
print('Дубликаты в participants:', participants.duplicated().sum())
print('Дубликаты в events:', events.duplicated().sum())

Дубликаты в participants: 0
Дубликаты в events: 36318


In [12]:
display(participants.describe(include='all'))

display(events.describe(include='all', datetime_is_numeric=True))

,user_id,group,ab_test,device
count,14525,14525,14525,14525
unique,13638,2,2,4
top,DBF9BDC859BC4459,A,interface_eu_test,Android
freq,2,8130,10850,6484


,user_id,event_dt,event_name,details
count,787286,787286,787286,249022
unique,144184,NaN,8,169
top,GLOBAL,NaN,login,4.99
freq,58495,NaN,248285,56063
mean,NaN,2020-12-16 18:02:17.995431424,NaN,NaN
min,NaN,2020-12-01 00:00:00,NaN,NaN
25%,NaN,2020-12-11 06:27:25.750000128,NaN,NaN
50%,NaN,2020-12-16 14:15:16.500000,NaN,NaN
75%,NaN,2020-12-22 04:28:38.249999872,NaN,NaN
max,NaN,2020-12-31 23:59:48,NaN,NaN


### Результаты первичной проверки

Таблица `participants` содержит 14 525 строк и 13 638 уникальных пользователей. Пропуски и полные дубликаты отсутствуют, однако часть пользователей участвует более чем в одном тесте или представлена в разных группах.

Таблица `events` содержит 787 286 событий. Поле `event_dt` приведено к формату `datetime`. В логах обнаружены полные дубликаты; перед удалением они рассматриваются как потенциальная особенность системы логирования. Поле `details` заполнено не для всех типов событий, что соответствует структуре данных.

In [13]:
# Полные совпадения в логах исключаются после проверки их происхождения.
events = events.drop_duplicates().reset_index(drop=True)
print('Количество событий после удаления полных дубликатов:', len(events))

Дубликаты в events после удаления: 0
Размер events после удаления дубликатов: (750968, 4)


## 3. Проверка состава участников и пересечений

In [14]:
display(participants.head())
display(participants.columns)

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


Index(['user_id', 'group', 'ab_test', 'device'], dtype='object')

In [15]:
user_col = 'user_id'
group_col = 'group'
test_col = 'ab_test'

display(participants[test_col].value_counts())
display(participants.groupby([test_col, group_col])[user_col].nunique())

interface_eu_test          10850
recommender_system_test     3675
Name: ab_test, dtype: int64

ab_test                  group
interface_eu_test        A        5383
                         B        5467
recommender_system_test  A        2747
                         B         928
Name: user_id, dtype: int64

In [16]:
# Выбираем изучаемый тест.
target_test = 'interface_eu_test'

In [17]:
test_participants = participants[participants[test_col] == target_test].copy()

display(test_participants.head())
display(test_participants[group_col].value_counts())
display(test_participants.groupby(group_col)[user_col].nunique())

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
2,001064FEAAB631A1,A,interface_eu_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac
5,002412F1EB3F6E38,B,interface_eu_test,Mac
6,002540BE89C930FB,B,interface_eu_test,Android


B    5467
A    5383
Name: group, dtype: int64

group
A    5383
B    5467
Name: user_id, dtype: int64

In [18]:
# Проверяем, нет ли пользователей, попавших сразу в две группы внутри изучаемого теста

users_in_several_groups = (
    test_participants
    .groupby(user_col)[group_col]
    .nunique()
    .reset_index()
    .query(f'{group_col} > 1')
)

display(users_in_several_groups.head())
print('Пользователей в нескольких группах изучаемого теста:', users_in_several_groups.shape[0])

,user_id,group


Пользователей в нескольких группах изучаемого теста: 0


In [19]:
# Удаляем пользователей, которые попали сразу в несколько групп изучаемого теста

bad_users = users_in_several_groups[user_col]

test_participants_clean = test_participants[
    ~test_participants[user_col].isin(bad_users)
].copy()

display(test_participants_clean.groupby(group_col)[user_col].nunique())

group
A    5383
B    5467
Name: user_id, dtype: int64

In [20]:
# Проверяем пересечения с конкурирующими тестами

target_users = set(test_participants_clean[user_col])

other_tests = participants[
    (participants[test_col] != target_test) &
    (participants[user_col].isin(target_users))
].copy()

intersect_users = other_tests[user_col].unique()

print('Пользователей, участвующих ещё и в других тестах:', len(intersect_users))
display(other_tests.head())

Пользователей, участвующих ещё и в других тестах: 887


,user_id,group,ab_test,device
1,001064FEAAB631A1,B,recommender_system_test,Android
9,00341D8401F0F665,A,recommender_system_test,iPhone
26,0082295A41A867B5,A,recommender_system_test,iPhone
41,00E68F103C66C1F7,A,recommender_system_test,PC
46,00EFA157F7B6E1C4,A,recommender_system_test,Android


In [21]:
# Исключаем пользователей, которые участвовали в конкурирующих тестах

test_participants_clean = test_participants_clean[
    ~test_participants_clean[user_col].isin(intersect_users)
].copy()

group_size = (
    test_participants_clean
    .groupby(group_col)[user_col]
    .nunique()
    .reset_index(name='users')
)

group_size['share'] = group_size['users'] / group_size['users'].sum()

display(group_size)

,group,users,share
0,A,4952,0.497039
1,B,5011,0.502961


### Результаты проверки участников

Для анализа выбран `interface_eu_test`, соответствующий эксперименту с изменением интерфейса. Первоначально в нём участвовали 10 850 пользователей, распределённых почти поровну между группами A и B.

Пересечений между группами внутри изучаемого теста не обнаружено. При этом 887 пользователей одновременно участвовали в конкурирующем эксперименте `recommender_system_test`; они исключены из дальнейшего анализа. После очистки осталось 4 952 пользователя в группе A и 5 011 пользователей в группе B.

### 3.1. Формирование выборки пользовательских событий

In [22]:
events_test = events.merge(
    test_participants_clean[[user_col, group_col]],
    on=user_col,
    how='inner'
)

display(events_test.head())
display(events_test[group_col].value_counts())
print('Событий участников изучаемого теста:', events_test.shape[0])
print('Пользователей с событиями:', events_test[user_col].nunique())

,user_id,event_dt,event_name,details,group
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,A
1,5F506CEBEDC05D30,2020-12-07 01:25:14,login,NaN,A
2,5F506CEBEDC05D30,2020-12-07 01:25:47,login,NaN,A
3,5F506CEBEDC05D30,2020-12-09 12:40:49,login,NaN,A
4,5F506CEBEDC05D30,2020-12-09 12:40:49,product_page,NaN,A


B    35044
A    33030
Name: group, dtype: int64

Событий участников изучаемого теста: 68074
Пользователей с событиями: 9963


In [23]:
display(events_test['event_name'].value_counts())

login           24121
product_page    16371
registration     9963
purchase         8901
product_cart     8718
Name: event_name, dtype: int64

В анализ включены только события пользователей, прошедших проверку состава участников изучаемого A/B-теста.

### 3.2. Единый семидневный горизонт анализа

In [24]:
# Определяем дату регистрации.
# Если она есть в таблице participants, используем её.
# Если нет, считаем датой регистрации первое событие пользователя.

possible_reg_cols = [
    col for col in test_participants_clean.columns
    if 'reg' in col.lower() or 'first' in col.lower() or 'date' in col.lower()
]

possible_reg_cols

[]

In [25]:
if len(possible_reg_cols) > 0:
    reg_col = possible_reg_cols[0]
    test_participants_clean[reg_col] = pd.to_datetime(test_participants_clean[reg_col])
    
    events_test = events.merge(
        test_participants_clean[[user_col, group_col, reg_col]],
        on=user_col,
        how='inner'
    )
    
    events_test = events_test.rename(columns={reg_col: 'registration_dt'})
else:
    registration_dates = (
        events_test
        .groupby(user_col)['event_dt']
        .min()
        .reset_index()
        .rename(columns={'event_dt': 'registration_dt'})
    )
    
    events_test = events_test.merge(
        registration_dates,
        on=user_col,
        how='left'
    )

events_test['lifetime'] = events_test['event_dt'] - events_test['registration_dt']

events_test_7d = events_test[
    (events_test['lifetime'] >= pd.Timedelta(days=0)) &
    (events_test['lifetime'] < pd.Timedelta(days=7))
].copy()

display(events_test_7d.head())
print('Событий в первые 7 дней:', events_test_7d.shape[0])
print('Пользователей с событиями в первые 7 дней:', events_test_7d[user_col].nunique())

,user_id,event_dt,event_name,details,group,registration_dt,lifetime
0,5F506CEBEDC05D30,2020-12-06 14:10:01,registration,0.0,A,2020-12-06 14:10:01,0 days 00:00:00
1,5F506CEBEDC05D30,2020-12-07 01:25:14,login,NaN,A,2020-12-06 14:10:01,0 days 11:15:13
2,5F506CEBEDC05D30,2020-12-07 01:25:47,login,NaN,A,2020-12-06 14:10:01,0 days 11:15:46
3,5F506CEBEDC05D30,2020-12-09 12:40:49,login,NaN,A,2020-12-06 14:10:01,2 days 22:30:48
4,5F506CEBEDC05D30,2020-12-09 12:40:49,product_page,NaN,A,2020-12-06 14:10:01,2 days 22:30:48


Событий в первые 7 дней: 58692
Пользователей с событиями в первые 7 дней: 9963


In [26]:
display(events_test_7d.groupby(group_col)[user_col].nunique())
display(events_test_7d.groupby([group_col, 'event_name'])[user_col].nunique())

group
A    4952
B    5011
Name: user_id, dtype: int64

group  event_name  
A      login           4952
       product_cart    1395
       product_page    3107
       purchase        1377
       registration    4952
B      login           5010
       product_cart    1327
       product_page    3184
       purchase        1480
       registration    5011
Name: user_id, dtype: int64

События ограничены первыми семью днями после регистрации. Это обеспечивает сопоставимый период наблюдения для пользователей обеих групп.

### 3.3. Оценка достаточности выборки

Расчёт выполнен для базовой конверсии 30%, целевого значения 33%, мощности 80% и уровня значимости 5%.

In [27]:
baseline_conversion = 0.30
expected_conversion = 0.33
alpha = 0.05
power = 0.80

effect_size = proportion_effectsize(
    baseline_conversion,
    expected_conversion
)

power_analysis = NormalIndPower()
required_sample_per_group = power_analysis.solve_power(
    effect_size=effect_size,
    power=power,
    alpha=alpha,
    ratio=1,
    alternative='two-sided'
)
required_sample_per_group = int(np.ceil(required_sample_per_group))

print('Минимальный размер выборки на группу:', required_sample_per_group)

Минимальный размер выборки на группу: 3762


In [28]:
actual_sample = (
    test_participants_clean
    .groupby(group_col)[user_col]
    .nunique()
    .reset_index(name='users')
)

actual_sample['required_users'] = required_sample_per_group
actual_sample['sample_enough'] = actual_sample['users'] >= required_sample_per_group

display(actual_sample)

,group,users,required_users,sample_enough
0,A,4952,3762,True
1,B,5011,3762,True


### Вывод о размере выборки

Минимально необходимый размер составляет 3 762 пользователя на группу. После очистки в обеих группах осталось более 4 900 участников, поэтому объём выборки достаточен для проверки целевого эффекта.

## 4. Расчёт конверсии в покупку

In [29]:
# Определяем событие покупки

events_test_7d['event_name'].value_counts()

login           23405
product_page    13946
registration     9963
purchase         5837
product_cart     5541
Name: event_name, dtype: int64

In [30]:
purchase_event = 'purchase'

if purchase_event not in events_test_7d['event_name'].unique():
    possible_purchase_events = [
        event for event in events_test_7d['event_name'].unique()
        if any(word in str(event).lower() for word in ['purchase', 'payment', 'buy', 'order'])
    ]
    
    if len(possible_purchase_events) > 0:
        purchase_event = possible_purchase_events[0]
    else:
        raise ValueError('Не найдено событие покупки. Проверьте названия событий в event_name.')

purchase_event

'purchase'

In [31]:
buyers = (
    events_test_7d[events_test_7d['event_name'] == purchase_event]
    .groupby(group_col)[user_col]
    .nunique()
    .reset_index(name='buyers')
)

visitors = (
    test_participants_clean
    .groupby(group_col)[user_col]
    .nunique()
    .reset_index(name='visitors')
)

conversion = visitors.merge(buyers, on=group_col, how='left')
conversion['buyers'] = conversion['buyers'].fillna(0).astype(int)
conversion['conversion'] = conversion['buyers'] / conversion['visitors']

display(conversion)

,group,visitors,buyers,conversion
0,A,4952,1377,0.278069
1,B,5011,1480,0.295350


Конверсия в группе A составила 27,81%, в группе B — 29,54%. Абсолютный прирост равен 1,73 процентного пункта, что ниже целевого эффекта в 3 процентных пункта. Для оценки статистической значимости проводится односторонний z-тест для двух долей.

In [32]:
control_group = 'A'
test_group = 'B'

control_conversion = conversion.loc[
    conversion[group_col] == control_group, 'conversion'
].iloc[0]

test_conversion = conversion.loc[
    conversion[group_col] == test_group, 'conversion'
].iloc[0]

diff = test_conversion - control_conversion
relative_diff = diff / control_conversion

print(f'Конверсия в группе A: {control_conversion:.2%}')
print(f'Конверсия в группе B: {test_conversion:.2%}')
print(f'Абсолютное изменение: {diff:.2%}')
print(f'Относительное изменение: {relative_diff:.2%}')

if diff > 0:
    print('Предварительно конверсия в тестовой группе выше, чем в контрольной.')
elif diff < 0:
    print('Предварительно конверсия в тестовой группе ниже, чем в контрольной.')
else:
    print('Предварительно конверсия в группах одинаковая.')

Конверсия в группе A: 27.81%
Конверсия в группе B: 29.54%
Абсолютное изменение: 1.73%
Относительное изменение: 6.21%
Предварительно конверсия в тестовой группе выше, чем в контрольной.


## 5. Статистическая оценка результата

### Гипотеза о целевом приросте конверсии

Проверяются следующие гипотезы:

- **H₀:** разница конверсий между группами B и A не превышает 3 процентных пункта;
- **H₁:** конверсия группы B превышает конверсию группы A более чем на 3 процентных пункта.

Уровень статистической значимости — 5%.

In [33]:
conversion

,group,visitors,buyers,conversion
0,A,4952,1377,0.278069
1,B,5011,1480,0.295350


In [34]:
buyers_a = conversion.loc[conversion[group_col] == 'A', 'buyers'].iloc[0]
visitors_a = conversion.loc[conversion[group_col] == 'A', 'visitors'].iloc[0]

buyers_b = conversion.loc[conversion[group_col] == 'B', 'buyers'].iloc[0]
visitors_b = conversion.loc[conversion[group_col] == 'B', 'visitors'].iloc[0]

count = np.array([buyers_b, buyers_a])
nobs = np.array([visitors_b, visitors_a])

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    value=0.03,
    alternative='larger'
)

print(f'z-статистика: {z_stat:.4f}')
print(f'p-value: {p_value:.4f}')

z-статистика: -1.4036
p-value: 0.9198


In [36]:
alpha = 0.05

if p_value < alpha:
    print('Отвергаем нулевую гипотезу.')
    print('Есть статистически значимые основания считать, что конверсия в группе B выше, чем в группе A, более чем на 3 процентных пункта.')
else:
    print('Не отвергаем нулевую гипотезу.')
    print('Статистически значимых оснований считать, что конверсия в группе B выше, чем в группе A, более чем на 3 процентных пункта, нет.')

Не отвергаем нулевую гипотезу.
Статистически значимых оснований считать, что конверсия в группе B выше, чем в группе A, более чем на 3 процентных пункта, нет.


### Результаты статистической проверки

После очистки в анализ вошли 4 952 пользователя группы A и 5 011 пользователей группы B. Конверсия выросла с 27,81% до 29,54%, однако фактический прирост составил только 1,73 процентного пункта.

Для проверки целевого эффекта применён односторонний z-тест для двух долей. Полученное значение `p-value = 0.9198` значительно выше уровня значимости 0,05, поэтому оснований считать, что новый интерфейс обеспечил прирост более 3 процентных пунктов, нет.

## Итоговый вывод и рекомендация

A/B-тест не подтвердил достижение целевого эффекта. Тестовая версия интерфейса показала более высокую конверсию, но наблюдаемый прирост оказался меньше минимально значимого эффекта и не получил статистического подтверждения.

Внедрять новый интерфейс только на основании текущего эксперимента не рекомендуется. Целесообразно изучить отдельные этапы воронки, проверить влияние интерфейса на другие продуктовые метрики и при необходимости провести повторный тест с уточнённой гипотезой.